# 01h - Remoção Espectral de Fingerprint (BARS & MEAN)

Aplica os dois ataques de remoção de fingerprint de GAN no domínio da frequência (Wesselkamp et al., *Misleading Deep-Fake Detection with GAN Fingerprints*, DLS 2022) aos nossos dados.

- **BARS** (untargeted): zera uma borda de altas frequências do espectro (a grade de upsampling). Parâmetro: `width`.
- **MEAN** (targeted): aprende `média(espectro fake) − média(espectro real)` e subtrai. Parâmetro: `factor`.

Etapas: (1) ajusta e visualiza a digital do StyleGAN; (2) aplica os ataques em exemplos; (3) diagnóstico destruição-vs-qualidade por intensidade; (4) BARS por gerador do ArtiFact.

Saída: digital em `artifacts/mean_fingerprint/` e figuras em `reports/figures/sfr_*.png`. Sem treinar CNN — minutos.

In [ ]:
import sys, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import gaussian_filter
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

sys.path.insert(0, str(Path.cwd().parent / "notebooks_140k"))
from aug_utils import artifact_split

# --- BARS/MEAN no dominio da frequencia (Wesselkamp 2022) ---
def load_image(path, size=None):
    img = Image.open(path).convert("RGB")
    if size is not None:
        img = img.resize((size, size), Image.BICUBIC)
    return np.asarray(img, dtype=np.float32) / 255.0

def to_spectrum(img):
    return np.fft.fftshift(np.fft.fft2(img, axes=(0, 1)), axes=(0, 1))

def from_spectrum(spec):
    return np.real(np.fft.ifft2(np.fft.ifftshift(spec, axes=(0, 1)), axes=(0, 1)))

def attack_bars(img, width=10):
    h, w = img.shape[:2]
    mask = np.ones((h, w), np.float32)
    if width > 0:
        mask[:width, :] = 0; mask[-width:, :] = 0; mask[:, :width] = 0; mask[:, -width:] = 0
    return from_spectrum(to_spectrum(img) * mask[:, :, None])

def fit_mean_fingerprint(fake_dir, real_dir, size=256, max_images=1500):
    def mean_spec(folder):
        paths = sorted(p for p in Path(folder).iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"})[:max_images]
        acc = np.zeros((size, size, 3), np.complex128)
        for p in paths:
            acc += to_spectrum(load_image(p, size=size))
        return acc / len(paths)
    return (mean_spec(fake_dir) - mean_spec(real_dir)).astype(np.complex64)

def attack_mean(img, fingerprint, factor=1.0):
    return from_spectrum(to_spectrum(img) - factor * fingerprint)

def psnr(a, b):
    mse = np.mean((np.clip(a, 0, 1) - np.clip(b, 0, 1)) ** 2)
    return float("inf") if mse == 0 else 10.0 * np.log10(1.0 / mse)

PROJECT_ROOT = Path.cwd().resolve().parent
_envf = PROJECT_ROOT / "data_root.env"
DATA_ROOT = Path(_envf.read_text().strip()) if _envf.exists() else PROJECT_ROOT / "data"
RAW_140K     = DATA_ROOT / "raw" / "140k_faces" / "real_vs_fake" / "real-vs-fake"
ARTIFACT_DIR = DATA_ROOT / "raw" / "artifact_faces"
FP_DIR  = PROJECT_ROOT / "artifacts" / "mean_fingerprint"
FIGS_DIR = PROJECT_ROOT / "reports" / "figures"
FP_DIR.mkdir(parents=True, exist_ok=True); FIGS_DIR.mkdir(parents=True, exist_ok=True)

SIZE, N_FIT, N_DIAG, SEED = 256, 1500, 250, 42
rng = np.random.default_rng(SEED)
print("140k:", RAW_140K.exists(), "| ArtiFact:", ARTIFACT_DIR.exists())

## 1. A digital média do StyleGAN

`fingerprint = média(espectro StyleGAN-fake) − média(espectro CelebA-real)`, ajustada no treino do 140k. Como as duas classes vêm do mesmo dataset, o pipeline de compressão é idêntico — a digital isola o resíduo do gerador.

In [ ]:
FP_PATH = FP_DIR / f"stylegan_140k_{SIZE}.npy"
if FP_PATH.exists():
    fingerprint = np.load(FP_PATH)
    print("digital carregada de", FP_PATH.name)
else:
    print(f"ajustando a digital (size={SIZE}, ate {N_FIT} imgs/classe)...")
    fingerprint = fit_mean_fingerprint(RAW_140K / "train" / "fake", RAW_140K / "train" / "real",
                                       size=SIZE, max_images=N_FIT)
    np.save(FP_PATH, fingerprint)
    print("salva em", FP_PATH)
mag = np.abs(fingerprint)
print("shape:", fingerprint.shape, "| magnitude media/max:", f"{mag.mean():.4f} / {mag.max():.4f}")

In [ ]:
logmag = np.log1p(np.abs(fingerprint).mean(axis=2))
c = SIZE // 2
view = logmag.copy(); view[c-2:c+3, c-2:c+3] = np.nan

fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
im0 = ax[0].imshow(view, cmap="magma")
ax[0].set_title("Digital MEAN do StyleGAN (log-magnitude)\npicos fora do centro = artefatos periodicos")
ax[0].axis("off"); plt.colorbar(im0, ax=ax[0], fraction=0.046)
yy, xx = np.indices((SIZE, SIZE)); r = np.hypot(xx - c, yy - c).astype(int)
rad = np.bincount(r.ravel(), np.abs(fingerprint).mean(2).ravel()) / np.maximum(np.bincount(r.ravel()), 1)
ax[1].plot(rad[:SIZE // 2], lw=2)
ax[1].set_title("Perfil radial da digital"); ax[1].set_xlabel("frequencia radial"); ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS_DIR / "sfr_digital_stylegan.png", dpi=130, bbox_inches="tight")
plt.show()

## 2. Os dois ataques numa imagem

In [ ]:
def ssim_gray(a01, b01, sigma=1.5):
    a = a01.mean(2) if a01.ndim == 3 else a01
    b = b01.mean(2) if b01.ndim == 3 else b01
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    ma, mb = gaussian_filter(a, sigma), gaussian_filter(b, sigma)
    va = gaussian_filter(a * a, sigma) - ma ** 2
    vb = gaussian_filter(b * b, sigma) - mb ** 2
    cov = gaussian_filter(a * b, sigma) - ma * mb
    s = ((2 * ma * mb + C1) * (2 * cov + C2)) / ((ma ** 2 + mb ** 2 + C1) * (va + vb + C2))
    return float(s.mean())

def logspec(img01):
    g = img01.mean(2) if img01.ndim == 3 else img01
    return np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(g))))

sty_files = sorted((RAW_140K / "train" / "fake").glob("*.jpg"))
sample = [sty_files[i] for i in rng.choice(len(sty_files), 3, replace=False)]
BARS_W, MEAN_F = 24, 1.0

fig, axes = plt.subplots(3, 6, figsize=(17, 9))
for row, p in enumerate(sample):
    orig = load_image(p, size=SIZE)
    b = np.clip(attack_bars(orig, width=BARS_W), 0, 1)
    m = np.clip(attack_mean(orig, fingerprint, factor=MEAN_F), 0, 1)
    panels = [(orig, "original"),
              (b, f"BARS w={BARS_W}\nPSNR {psnr(orig, b):.1f} SSIM {ssim_gray(orig, b):.2f}"),
              (m, f"MEAN f={MEAN_F}\nPSNR {psnr(orig, m):.1f} SSIM {ssim_gray(orig, m):.2f}")]
    for col, (im, ttl) in enumerate(panels):
        axes[row, col].imshow(im); axes[row, col].set_title(ttl, fontsize=8); axes[row, col].axis("off")
        axes[row, col + 3].imshow(logspec(im), cmap="magma"); axes[row, col + 3].axis("off")
        if row == 0:
            axes[row, col + 3].set_title(["espectro orig", "espectro BARS", "espectro MEAN"][col], fontsize=8)
plt.suptitle("StyleGAN fake: imagem (esq) e espectro (dir) - original vs BARS vs MEAN")
plt.tight_layout()
plt.savefig(FIGS_DIR / "sfr_antes_depois.png", dpi=120, bbox_inches="tight")
plt.show()

## 3. Diagnóstico — destruição do atalho vs custo de qualidade

Aplica o ataque só nos fakes, mede a AUC de um probe espectral real-vs-fake (queda = atalho removido) e o SSIM (custo). O ponto ideal derruba a AUC perto de 0.5 mantendo SSIM alto.

In [ ]:
def radial_feat(img01):
    g = img01.mean(2) if img01.ndim == 3 else img01
    F = np.fft.fftshift(np.abs(np.fft.fft2(g))) ** 2
    rr = np.hypot(*[a - SIZE // 2 for a in np.indices((SIZE, SIZE))][::-1]).astype(int)
    prof = np.bincount(rr.ravel(), F.ravel()) / np.maximum(np.bincount(rr.ravel()), 1)
    return np.log1p(prof[:SIZE // 2])

real_files = sorted((RAW_140K / "train" / "real").glob("*.jpg"))
fk = [load_image(sty_files[i], size=SIZE) for i in rng.choice(len(sty_files), N_DIAG, replace=False)]
rl = [load_image(real_files[i], size=SIZE) for i in rng.choice(len(real_files), N_DIAG, replace=False)]
Xr = np.stack([radial_feat(im) for im in rl])

def probe_auc(fake_imgs):
    Xf = np.stack([radial_feat(im) for im in fake_imgs])
    X = np.concatenate([Xr, Xf]); y = np.r_[np.zeros(len(Xr)), np.ones(len(Xf))]
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    return cross_val_score(clf, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring="roc_auc").mean()

auc0 = probe_auc(fk)
rows = []
for w in [4, 8, 12, 16, 24, 32, 48, 64]:
    att = [np.clip(attack_bars(im, width=w), 0, 1) for im in fk]
    rows.append(("bars", w, probe_auc(att), np.mean([ssim_gray(o, a) for o, a in zip(fk, att)])))
for f in [0.25, 0.5, 0.75, 1.0, 1.5, 2.0]:
    att = [np.clip(attack_mean(im, fingerprint, factor=f), 0, 1) for im in fk]
    rows.append(("mean", f, probe_auc(att), np.mean([ssim_gray(o, a) for o, a in zip(fk, att)])))
diag = pd.DataFrame(rows, columns=["ataque", "valor", "auc", "ssim"])
print(f"AUC espectral real-vs-fake (StyleGAN), imagens limpas: {auc0:.3f}\n")
print(diag.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for atk, mk in [("bars", "o"), ("mean", "s")]:
    d = diag[diag.ataque == atk]
    axes[0].plot(d.valor, d.auc, marker=mk, label=atk)
    axes[1].plot(d.ssim, d.auc, marker=mk, label=atk)
axes[0].axhline(auc0, color="gray", ls="--", lw=1, label=f"limpa ({auc0:.2f})")
axes[0].axhline(0.5, color="red", ls=":", lw=1, label="chance")
axes[0].set_xlabel("intensidade (width / factor)"); axes[0].set_ylabel("AUC espectral"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[1].axhline(0.5, color="red", ls=":", lw=1)
axes[1].set_xlabel("SSIM (qualidade)"); axes[1].set_ylabel("AUC espectral"); axes[1].invert_xaxis()
axes[1].set_title("canto inferior direito = ideal"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS_DIR / "sfr_diagnostico.png", dpi=130, bbox_inches="tight")
plt.show()

ok = diag[diag.ssim >= 0.85]
if len(ok):
    best = ok.loc[ok.auc.idxmin()]
    print(f"Sugestao (SSIM>=0.85): {best.ataque} = {best.valor} -> AUC {best.auc:.3f}, SSIM {best.ssim:.2f}")

## 4. BARS cross-generator (untargeted)

O BARS não precisa de digital ajustada. Aplica a cada gerador do ArtiFact (metade `dev`) e mede a queda da AUC espectral.

In [ ]:
def list_by_source(folder):
    g = {}
    for p in folder.glob("*.*"):
        s = p.name.split("__")[0] if "__" in p.name else "?"
        g.setdefault(s, []).append(p)
    return {k: sorted(v) for k, v in g.items()}

fake_groups = list_by_source(ARTIFACT_DIR / "fake")
real_groups = list_by_source(ARTIFACT_DIR / "real")
GEN = sorted(fake_groups)

art_real = []
for s in sorted(real_groups):
    dev = artifact_split(real_groups[s], which="dev")
    art_real += [load_image(dev[i], size=SIZE) for i in rng.choice(len(dev), 30, replace=False)]
Xar = np.stack([radial_feat(im) for im in art_real])

def auc_gen(fimgs):
    Xf = np.stack([radial_feat(im) for im in fimgs])
    X = np.concatenate([Xar, Xf]); y = np.r_[np.zeros(len(Xar)), np.ones(len(Xf))]
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    return cross_val_score(clf, X, y, cv=StratifiedKFold(4, shuffle=True, random_state=SEED), scoring="roc_auc").mean()

W = 24
recs = []
for g in GEN:
    dev = artifact_split(fake_groups[g], which="dev")
    imgs = [load_image(dev[i], size=SIZE) for i in rng.choice(len(dev), 80, replace=False)]
    a0 = auc_gen(imgs)
    aB = auc_gen([np.clip(attack_bars(im, width=W), 0, 1) for im in imgs])
    recs.append({"gerador": g, "auc_limpa": round(a0, 3), f"auc_bars_w{W}": round(aB, 3), "delta": round(aB - a0, 3)})
print(f"BARS (width={W}) por gerador - queda da AUC espectral:\n")
print(pd.DataFrame(recs).sort_values("delta").to_string(index=False))

## 5. Leitura e próximos passos

- A digital (Seção 1): picos fora do centro = a grade de upsampling do StyleGAN, o atalho.
- O diagnóstico (Seção 3): o ponto da curva que derruba a AUC perto de 0.5 com SSIM alto é a intensidade certa de cada ataque.
- BARS cross-generator (Seção 4): forte nos GANs com grade, fraco na difusão.

**Onde cada um vai:** BARS (`width` ótimo) → pool de augmentation (`aug_utils`); MEAN (`factor` ótimo) → augmentation targeted no treino, removendo a digital salva dos fakes. Confirmar o ganho cross-generator no `01i_treino_cross_generator`.